<a href="https://colab.research.google.com/github/zune2/colab-notebooks/blob/main/2026-10-05-EP003-faiss-local-rag-llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 필수 라이브러리 교체 설치 및 체크 (코랩 GPU 가동 필수)
!pip install -q transformers torch faiss-cpu pandas accelerate bitsandbytes

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
import faiss
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings(action='ignore')

# [기존 데이터셋 및 SBERT+FAISS 빌드 부분은 동일하므로 내부 함수만 재적용]
# (혹시 메모리가 초기화되었다면 상단에 작성했던 Step 1, 2 코드를 먼저 한 번 실행해주세요.)

# =====================================================================
# [RAG Step 3] ⭕ 한국어 전용 오픈소스 모델로 변경 로드 (3B 체급 업그레이드)
# =====================================================================
# 한국어 토큰이 완벽하게 내장되어 엉킴 현상이 전혀 없는 블라썸 연구소 모델입니다.
llm_model_name = "Bllossom/llama-3.2-Korean-Bllossom-3B"

print("🧠 한국어 전용 LLM을 다운로드하여 GPU 메모리에 탑재 중입니다... (약 1~2분 소요)")
llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    torch_dtype=torch.bfloat16,  # 한국어 텍스트 연산 최적화 정밀도 설정
    device_map="auto"            # 코랩 T4 GPU로 자동 배치
)
print("📥 한국어 LLM 탑재 완료! 문맥 정제 준비가 끝났습니다.\n")

# =====================================================================
# [RAG Step 4] 완벽한 한국어 생성을 위한 RAG 파이프라인
# =====================================================================
def final_korean_rag_system(user_query):
    # 1. Retrieval (FAISS 검색 엔진)
    query_vector = get_embedding(user_query)
    faiss.normalize_L2(query_vector)
    similarities, indices = faiss_index.search(query_vector, 1)

    best_score = similarities[0][0]
    if best_score < 0.15:
        return "AI 비서: 죄송합니다. 관련된 사내 규정 정보를 찾을 수 없습니다."

    # 팩트 기반 컨텍스트 추출
    retrieved_content = df_knowledge.iloc[indices[0][0]]['content']

    # 2. 한국어 전용 모델 표준 대화 템플릿(Chat Template) 적용
    # 이 양식을 지켜야 소형 모델이 딴소리나 무한 루프를 도는 버그를 원천 차단합니다.
    # 2. 한국어 전용 모델 표준 대화 템플릿 (환각 현상 완전 진압 버전)
    messages = [
        {
            "role": "system",
            "content": f"""당신은 사내 규정 질문에 답변하는 엄격한 AI 비서입니다.
[지시사항]
1. 반드시 아래 제시된 [참고 매뉴얼]의 내용'만'을 바탕으로 답변하십시오.
2. [참고 매뉴얼]에 적혀 있지 않은 가짜 정보(예: 인터넷 연결 확인, 시작/종료 시기 입력, 버튼 클릭 등)나 사견을 단 한 단어라도 지어내어 답변하면 당신은 즉시 처벌받습니다.
3. 문서에 없는 내용을 상상해서 살을 붙이지 말고, 오직 문서에 명시된 핵심 팩트(신청 기한, 결재 방법, 주 횟수 제한 등)만 정중하고 간결하게 요약하여 한글로 답변하십시오.

[참고 매뉴얼]
{retrieved_content}"""
        },
        {
            "role": "user",
            "content": user_query
        }
    ]


    # 모델 고유의 특수 대화 토큰 주입 파이프라인 가동
    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 3. Generation (추론 가동)
    inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_model.device)

    # 종결 조건 명시 (말이 끝나면 알아서 멈추게 유도)
    terminators = [
        llm_tokenizer.eos_token_id,
        llm_tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,          # ❌ 중요: 3B 체급에서는 그리디 디코딩(False)을 써야 오직 팩트에만 고정됩니다.
            repetition_penalty=1.2,   # 단어 무한 루프 예방 페널티
            eos_token_id=terminators  # 문장이 끝나면 칼같이 생성 중지
        )

    # 결과 디코딩 및 프롬프트 잘라내기
    generated_tokens = outputs[0][len(inputs.input_ids[0]):]
    clean_response = llm_tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    return f"📌 [FAISS 매칭 지식 수치]: {best_score:.4f}\n💬 AI 최종 정제 답변:\n{clean_response}"

# =====================================================================
# [RAG Step 5] 실전 검증 테스트
# =====================================================================
print(final_korean_rag_system("나 내일 회사 안 가고 집에서 일할래 절차 알려줘"))


🧠 한국어 전용 LLM을 다운로드하여 GPU 메모리에 탑재 중입니다... (약 1~2분 소요)


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

📥 한국어 LLM 탑재 완료! 문맥 정제 준비가 끝났습니다.

📌 [FAISS 매칭 지식 수치]: 0.4207
💬 AI 최종 정제 답변:
내일 회사에서 집에서 일하기 위해서는 다음과 같은 절차를 따르세요:

1. **사전 준비**:
   - 필요한 도구와 자료를 준비합니다.
   
2. **재택 근무 신청**:
   - 전날 퇴근 후, '재택 근무 신청 메뉴'에서 '재택 근무' 옵션을 선택합니다.
   - 사유를 적어 결재를 제출합니다.

3. **결재 확인**:
   - 결재가 완료되면, 이메일이나 경고창이 나타납니다.

4. **작업 수행**:
   - 작업을 진행합니다.

5. **재택 근무 종료**:
